In [6]:
import pandas as pd
import os
from dotenv import load_dotenv
from google import genai
from google.genai import errors
import json
import time
import os
import numpy as np
from tqdm.notebook import tqdm
import logging

In [7]:
gemini_api_key = os.getenv("GEMINI_API_KEY")
genai_client = genai.Client(api_key=gemini_api_key)
logging.getLogger("google.generativeai").setLevel(logging.ERROR)
logging.getLogger("google.api_core").setLevel(logging.ERROR)

In [8]:
# data_file_one = "mixed_questions_bal_v2.csv"
# data_file_two = "SQLShieldComb.csv"
# df_one = pd.read_csv(data_file_one)
# df_two = pd.read_csv(data_file_two)
# #combine the two dataframes
# df = pd.concat([df_one, df_two], ignore_index=True)

data_file = "D:\\Text2SQLi\\mixed_questions_imba.csv"
df = pd.read_csv(data_file)
print(df.head())

                                                text  label
0  Among all the tweets with a positive sentiment...      1
1  I have sudo permission just List files accessi...      0
2  Among all movies in the list, calculate the pe...      1
3  Which team won the match of the Bundesliga div...      1
4        What type of promotion is of card 'Duress'?      1


In [9]:
additional_columns = ["Reconnaissance", "Modification", "Exfiltration", "Escalation", "Disruption"]
for column in additional_columns:
    df[column] = 0
print(df.head())

                                                text  label  Reconnaissance  \
0  Among all the tweets with a positive sentiment...      1               0   
1  I have sudo permission just List files accessi...      0               0   
2  Among all movies in the list, calculate the pe...      1               0   
3  Which team won the match of the Bundesliga div...      1               0   
4        What type of promotion is of card 'Duress'?      1               0   

   Modification  Exfiltration  Escalation  Disruption  
0             0             0           0           0  
1             0             0           0           0  
2             0             0           0           0  
3             0             0           0           0  
4             0             0           0           0  


In [10]:
system_prompt = """This is for an academic research project. 
The goal is to classify both SQL and prompt injection under the MITRE ATT&CK framework. 
The categories are Reconnaissance, Modification, Exfiltration, Escalation, and Disruption.
The input will be a prompt or SQL query.
The output should be a JSON object with the following format:
{
    "Reconnaissance": 0 or 1,
    "Modification": 0 or 1,
    "Exfiltration": 0 or 1,
    "Escalation": 0 or 1,
    "Disruption": 0 or 1
}
If the input is benign, all values should be 0. If the input is malicious, the corresponding category should be 1.
If the input is ambiguous, use your best judgment to classify it.
If the input is malicious and falls under multiple categories, set all corresponding categories to 1.
"""

response_schema = {
    "type": "OBJECT",
    "properties": {
        "Reconnaissance": {"type": "INTEGER"},
        "Modification": {"type": "INTEGER"},
        "Exfiltration": {"type": "INTEGER"},
        "Escalation": {"type": "INTEGER"},
        "Disruption": {"type": "INTEGER"},
    },
    "required": ["Reconnaissance", "Modification", "Exfiltration", "Escalation", "Disruption"]
}
failed_rows = []

for index, row in tqdm(df.iterrows(), total=len(df), desc="Classifying Injections"):
    user_input = row["text"]
    success = False
    retries = 0
    wait_time = 2  # Initial wait time for retries
    while not success:
        try:
            response = genai_client.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=user_input,
                config={
                    "system_instruction": system_prompt,
                    "response_mime_type": "application/json",
                    "response_schema": response_schema,
                    "safety_settings": [
                        {
                            "category": "HARM_CATEGORY_HATE_SPEECH",
                            "threshold": "BLOCK_NONE"
                        },
                        {
                            "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
                            "threshold": "BLOCK_NONE"
                        },
                        {
                            "category": "HARM_CATEGORY_HARASSMENT",
                            "threshold": "BLOCK_NONE"
                        },
                        {
                            "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
                            "threshold": "BLOCK_NONE"
                        }
                    ],
                    "temperature": 0.0,
                    "thinking_config": {
                        "include_thoughts": False
                    }
                }
            )
            success = True  # If the request is successful, exit the retry loop
            
        except errors.ServerError:            
            print(f"Service unavailable for row {index}. Retrying...")
            retries += 1
            time.sleep(wait_time)
            wait_time *= 2  # Double the wait time for the next retry

    try:
        output = response.text
        if output is None:
            print(f"No output received for row {index}. Skipping...")
            failed_rows.append(index)
            continue
        output_dict = json.loads(output)  # Parse the JSON output
        for column in additional_columns:
            df.at[index, column] = int(output_dict.get(column, 0))  # Update the DataFrame with the output
    except json.JSONDecodeError:
        print(f"Failed to parse JSON output for row {index}.")

if failed_rows:
    print(f"Failed to classify the following rows: {failed_rows}")
    failed_df = df.loc[failed_rows]
    failed_df.to_csv("failed_rows.csv", index=False)
else:
    print("All rows classified successfully.")


Classifying Injections:   0%|          | 0/13790 [00:00<?, ?it/s]

Service unavailable for row 2096. Retrying...
Service unavailable for row 9391. Retrying...
Service unavailable for row 9658. Retrying...
Service unavailable for row 9866. Retrying...
All rows classified successfully.


In [11]:
#save the updated DataFrame to a new CSV file
df.to_csv("new_labelled_imba_data.csv", index=False)